# nvidia_ml_py smoke test

Tests `pynvml` — Python bindings to the NVIDIA Management Library (NVML).

Notes:
- The import name is `pynvml` (the distribution name is `nvidia-ml-py`).
- The package is hybrid: it uses `ctypes` to load `libnvidia-ml.so.1` at runtime.
- If no NVIDIA driver is present, `nvmlInit()` raises `NVMLError_LibraryNotFound`/`NVMLError_DriverNotLoaded`. Tests below tolerate that.

In [ ]:
# Public-API surface check: a handful of canonical NVML symbols must be present.
import pynvml

expected = [
    'nvmlInit',
    'nvmlShutdown',
    'nvmlDeviceGetCount',
    'nvmlDeviceGetHandleByIndex',
    'nvmlDeviceGetName',
    'nvmlDeviceGetMemoryInfo',
    'nvmlSystemGetDriverVersion',
    'NVMLError',
]
missing = [name for name in expected if not hasattr(pynvml, name)]
print(f"missing: {missing}")
assert not missing, f"pynvml is missing expected NVML symbols: {missing}"
print(f"OK: all {len(expected)} canonical NVML symbols present")

In [ ]:
# nvmlInit / nvmlShutdown lifecycle. Tolerates absence of NVIDIA driver.
import pynvml

have_driver = False
try:
    pynvml.nvmlInit()
    have_driver = True
    driver_version = pynvml.nvmlSystemGetDriverVersion()
    if isinstance(driver_version, bytes):
        driver_version = driver_version.decode('utf-8', errors='replace')
    print(f"NVML initialized; driver version = {driver_version}")
    pynvml.nvmlShutdown()
    print("NVML shutdown clean")
except pynvml.NVMLError as e:
    # Expected on hosts without an NVIDIA driver (CI, dev containers, etc.).
    print(f"NVML unavailable (expected if no NVIDIA driver): {type(e).__name__}: {e}")

In [ ]:
# Device enumeration. If NVML is unavailable, the call raises an NVMLError.
import pynvml

try:
    pynvml.nvmlInit()
    try:
        count = pynvml.nvmlDeviceGetCount()
        print(f"NVML device count = {count}")
        for i in range(count):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            name = pynvml.nvmlDeviceGetName(handle)
            if isinstance(name, bytes):
                name = name.decode('utf-8', errors='replace')
            print(f"  [{i}] {name}")
    finally:
        pynvml.nvmlShutdown()
except pynvml.NVMLError as e:
    print(f"NVML unavailable (expected if no NVIDIA driver): {type(e).__name__}: {e}")

In [ ]:
# Error-class hierarchy: NVMLError subclasses are correctly registered.
import pynvml

# Every concrete NVMLError_* subclass must inherit from NVMLError and carry a value.
error_subclasses = [
    name for name in dir(pynvml)
    if name.startswith('NVMLError_') and isinstance(getattr(pynvml, name), type)
]
print(f"Found {len(error_subclasses)} NVMLError_* subclasses")
assert error_subclasses, "expected at least a few NVMLError_* subclasses"

for name in error_subclasses[:5]:
    cls = getattr(pynvml, name)
    assert issubclass(cls, pynvml.NVMLError), f"{name} is not an NVMLError subclass"
    print(f"  {name} <- NVMLError (value={getattr(cls, 'value', None)})")

# Raising + catching round-trip.
try:
    raise pynvml.NVMLError_NotSupported()
except pynvml.NVMLError as e:
    print(f"\nCaught {type(e).__name__} as NVMLError: OK")

assert issubclass(pynvml.NVMLError_NotSupported, pynvml.NVMLError)

In [ ]:
# Constants & ctypes-backed struct types are present and well-formed.
import ctypes

import pynvml

# A few canonical constants from nvml.h.
assert hasattr(pynvml, 'NVML_SUCCESS'), 'NVML_SUCCESS constant missing'
assert pynvml.NVML_SUCCESS == 0, f'NVML_SUCCESS should be 0, got {pynvml.NVML_SUCCESS}'

# Temperature sensor enum.
assert hasattr(pynvml, 'NVML_TEMPERATURE_GPU')
print(f"NVML_TEMPERATURE_GPU = {pynvml.NVML_TEMPERATURE_GPU}")

# c_nvmlMemory_t / nvmlMemory_t — the ctypes Structure used by nvmlDeviceGetMemoryInfo.
mem_struct_name = next(
    (n for n in ('c_nvmlMemory_t', 'nvmlMemory_t') if hasattr(pynvml, n)),
    None,
)
assert mem_struct_name is not None, 'no nvmlMemory_t ctypes struct on pynvml'
mem_struct = getattr(pynvml, mem_struct_name)
assert issubclass(mem_struct, ctypes.Structure), f'{mem_struct_name} is not a ctypes.Structure'
field_names = {f[0] for f in mem_struct._fields_}
print(f"{mem_struct_name}._fields_ = {sorted(field_names)}")
assert {'total', 'free', 'used'}.issubset(field_names), \
    f'expected total/free/used in {mem_struct_name}, got {field_names}'
print('OK: NVML constants + nvmlMemory_t struct surface look correct')